# Core claims and argument maps v02: resolve restated ideas first

v01 (`argument_core_test.ipynb`) let the models pick among atoms that restate the same idea (e.g. "The Fed hikes 25bp" appears in P005, P012, P135). v02 adds a **resolution step**:

1. **Resolve** (Opus, once, cached): cluster atoms that express the same idea (neither could be true while the other is false; same polarity and modality class). Each cluster keeps one canonical atom, and the others are recorded as "also stated as".
2. **Map** (Haiku, Sonnet, Opus x `nocore`/`core`): same task as v01, but over the canonical atoms only. Core flags carry over: a canonical atom is core if any member comes from a core proposition (union of Haiku and Opus votes in `claim_core_test_v02`).
3. **Compare** with v01 on repeated ideas (atoms in a map that belong to the same cluster) and idea-level agreement across models.

Outputs: `outputs/argument_core_test_v02/` (`relations.json`, then `<condition>/<model>/` with `argument_map.json`, `argument_map.yaml`, themed `diagram.mmd`). Delete a file or folder to re-run it.

In [ ]:
import json
import re
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from incolor.argumentation.backends.claude_agent import ClaudeAgentBackend

OUT = ROOT / "outputs" / "argument_core_test_v02"
OUT.mkdir(parents=True, exist_ok=True)
V01 = ROOT / "outputs" / "argument_core_test"
MODELS = ["haiku", "sonnet", "opus"]
CONDITIONS = ["nocore", "core"]
ROLES = ["premises", "conditions", "qualifiers", "concessions", "risks", "implications", "alternatives"]

DOC = re.sub(r"\A---.*?---\s*", "", (ROOT / "samples/BofA_Sept_FOMC_-_hike_or_bond_spike.txt").read_text(encoding="utf-8"), flags=re.S)
ATOMISE = ROOT / "outputs/atomise_v01/opus"
props = {
    m.group(1): m.group(2)
    for ln in (ATOMISE / "input.txt").read_text(encoding="utf-8").splitlines()
    if (m := re.match(r"\[(P\d+)\]\s+(.*)", ln))
}
ALL_ATOMS = {a["id"]: a for a in (json.loads(ln) for ln in (ATOMISE / "atoms.jsonl").read_text(encoding="utf-8").splitlines() if ln.strip())}
for a in ALL_ATOMS.values():
    a["pid"] = a["id"].split("-")[0]

# Core propositions: union of the Haiku and Opus votes
CORE_PROPS = sorted(
    {
        row["id"]
        for m in ("haiku", "opus")
        for row in json.loads((ROOT / f"outputs/claim_core_test_v02/{m}.json").read_text(encoding="utf-8"))["rows"]
        if row.get("core")
    }
)
print(len(props), "propositions,", len(ALL_ATOMS), "atoms; core propositions:", CORE_PROPS)

120 propositions, 269 atoms; core propositions: ['P005', 'P007', 'P009', 'P010']


In [3]:
RESOLVE_SYSTEM = """You resolve repeated ideas in a sell-side FX/rates research note. You get the full note, then its claims pre-extracted as atoms (IDs like P009-2, each with an extractor modality tag), grouped under the sentence they came from. The note often states the same idea more than once (key takeaways, body, summary).

Task: cluster atoms that express THE SAME IDEA, so each idea is represented once.

Two atoms are the same idea only if, in the context of this note, neither could be true while the other is false: same subject, same predicate, same polarity, same time and scope. Differences in wording, pronouns, tense, or trivial detail do not matter.

Do NOT merge:
- atoms that are merely related: cause and effect, premise and conclusion, part and whole, general and specific, or the same topic with different content (e.g. "hikes 25bp" vs "hikes 75bp in total");
- atoms of different modality class: an author's expectation, a plain assertion, a supposed condition (SUPPOSED) and a claim attributed to others are different even when the wording matches (e.g. EXPECTED "The Fed hikes 25bp" is not the same idea as SUPPOSED "The Fed hikes");
- atoms in different scenarios whose conditions or consequences differ.

For each cluster choose the canonical atom: the clearest, most complete wording, preferring the headline statement (key takeaways) when it is equally clear. List every member, including the canonical atom. Report only clusters with 2 or more members; atoms not listed stay on their own. Each atom belongs to at most one cluster.

Return ONE JSON object and nothing else (no code fences):
{"clusters": [{"canonical": "P005-1", "members": ["P005-1", "P135-1"], "why": "<=10 words"}]}"""


def atom_blocks() -> str:
    blocks = []
    for pid, text in props.items():
        group = [a for a in ALL_ATOMS.values() if a["pid"] == pid]
        if group:
            blocks.append(f"[{pid}] {text}\n" + "\n".join(f"  {a['id']} [{a['modality']}] {a['text']}" for a in group))
    return "\n".join(blocks)


def validate_clusters(data: dict) -> list[str]:
    errors, seen = [], Counter()
    for c in data.get("clusters", []):
        members = c.get("members", [])
        if len(members) < 2:
            errors.append(f"cluster {c.get('canonical')!r} has fewer than 2 members")
        if c.get("canonical") not in members:
            errors.append(f"canonical {c.get('canonical')!r} is not among its members")
        for x in members:
            if x not in ALL_ATOMS:
                errors.append(f"unknown atom {x!r}")
            seen[x] += 1
    return errors + [f"atom {x} is in {n} clusters" for x, n in seen.items() if n > 1]


def resolve() -> dict:
    path = OUT / "relations.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    user = f"<document>\n{DOC}\n</document>\n\n<atoms>\n{atom_blocks()}\n</atoms>"
    backend = ClaudeAgentBackend(model="opus")
    raws, errors, data = [], [], {}
    for attempt in range(2):
        prompt = user if attempt == 0 else user + "\n\nYour previous answer had these errors; return the corrected full JSON object:\n- " + "\n- ".join(errors[:30]) + f"\n\nPrevious answer:\n{raws[-1]}"
        raws.append(backend.generate(RESOLVE_SYSTEM, prompt, {}))
        try:
            data = json.loads(raws[-1][raws[-1].index("{") : raws[-1].rindex("}") + 1])
            errors = validate_clusters(data)
        except ValueError as exc:
            data, errors = {}, [f"unparseable JSON: {exc}"]
        if not errors:
            break
    result = {"clusters": data.get("clusters", []), "errors": errors, "raw": raws, "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


# The Claude backend calls asyncio.run, so it must not run on the notebook's event-loop thread
with ThreadPoolExecutor(1) as pool:
    relations = pool.submit(resolve).result()
assert not relations["errors"], relations["errors"]

# cluster_of maps every atom to its canonical atom; atoms (canonical only) is what the map builders see
cluster_of = {i: i for i in ALL_ATOMS}
for c in relations["clusters"]:
    for x in c["members"]:
        cluster_of[x] = c["canonical"]
atoms = {}
for i, a in ALL_ATOMS.items():
    if cluster_of[i] == i:
        members = [x for x, k in cluster_of.items() if k == i]
        atoms[i] = {**a, "members": members, "pids": {ALL_ATOMS[x]["pid"] for x in members}}
CORE_ATOMS = {i for i, a in atoms.items() if a["pids"] & set(CORE_PROPS)}

merged = [c for c in relations["clusters"]]
print(f"{len(ALL_ATOMS)} atoms -> {len(atoms)} ideas ({len(merged)} merged clusters, {len(ALL_ATOMS) - len(atoms)} atoms folded away); {len(CORE_ATOMS)} core ideas")
cost = sum(u.get("cost_usd") or 0 for u in relations["usage"])
print(f"resolution cost ${cost:.3f}")
for c in merged:
    print(f"  {c['canonical']}: {ALL_ATOMS[c['canonical']]['text']}  <- {', '.join(x for x in c['members'] if x != c['canonical'])}")

269 atoms -> 247 ideas (13 merged clusters, 22 atoms folded away); 7 core ideas
resolution cost $0.232
  P005-1: The Fed hikes 25bp.  <- P135-1
  P022-1: The SEP shows 50bps of total hikes in '26.  <- P136-1
  P016-1: The Fed faces a simple choice at the Sept FOMC.  <- P007-1, P137-1, P098-1
  P016-2: The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike.  <- P007-2, P137-2, P098-2
  P019-1: Bond price action, including post Friday's CPI, shows the bond likes orthodoxy.  <- P008-1
  P009-1: Warsh delivers a hawkish hike.  <- P138-1, P102-1, P085-1
  P009-2: Warsh's hawkish hike supports higher front end rates.  <- P138-2, P103-3
  P009-3: Warsh's hawkish hike supports twist flattening of the UST curve.  <- P138-3, P103-4, P081-1
  P010-1: The USD outcome is determined by the Fed's ability to "out-hawk" market expectations and other G10 central banks.  <- P139-1
  P013-1: Rates price the Sep hike with approximately 85% likelihood.  <- P079-1
  P096-12: Treas

In [4]:
SYSTEM = """You are a financial-research argument analyst. You get a sell-side research note in full, then its claims pre-extracted as atoms (IDs like P009-2, each with the extractor's modality tag), grouped under the sentence they came from. Atoms that restate the same idea have already been merged: each idea appears once, and an "also stated as" note shows where else the note repeats it.

Task: rebuild the CORE ARGUMENTS of the note as an argument map over the atoms. Do not use every atom: use only the atoms that do argumentative work for the author's main conclusions, and leave out the rest (background, data not needed as support, side scenarios, detail).

An argument has ONE conclusion (an atom ID) and the members that bear on it, each an atom ID or the ID of another argument (A###, meaning that argument's conclusion is itself a member, supported by its own members). Roles:
- premises: reasons or evidence the author gives for the conclusion
- conditions: what the conclusion depends on (if/unless/assuming)
- qualifiers: scope or strength limits on the conclusion
- concessions: points the author grants that cut against the conclusion
- risks: what could make the conclusion fail or hurt
- implications: what follows from the conclusion
- alternatives: the author's stated alternative outcome

Rules:
- Use the whole note to see how the author argues; link members only where the author states or clearly structures the link. Do not invent links.
- Use atom IDs exactly as given. Never rewrite atom text.
- Each atom appears at most once in the whole map (as one argument's conclusion or as one member of one argument).
- Never restate an idea: a point that is already a conclusion or member must not reappear in another role or argument. Use the "also stated as" locations only to understand where the author argues the point.
- Each argument is a member of at most one other argument; no cycles. Top-level arguments are those nobody references.
- Every argument has at least one premise.
- Build hierarchy: if a premise has reasons of its own, make it the conclusion of its own argument and reference that argument's ID as the member.
- Summaries: one sentence, at most 30 words, using only claims of the member atoms.

Return ONE JSON object and nothing else (no code fences):
{"arguments": [{"id": "A001", "summary": "...", "conclusion": "P005-1", "premises": ["P012-1", "A002"], "conditions": [], "qualifiers": [], "concessions": [], "risks": [], "implications": [], "alternatives": []}]}
Number arguments A001, A002, ... in the order you list them."""

CORE_BLOCK = """
The author's headline (core) claims have been identified in a prior pass and are flagged CORE in the atom list. Build the map so that each top-level argument concludes one of the core claims, or the core claims are reached through the hierarchy. Reasoning that does not bear on a core claim need not appear."""


def build_user(condition: str) -> str:
    blocks = []
    for pid, text in props.items():
        group = [a for a in atoms.values() if a["pid"] == pid]
        if not group:
            continue
        lines = []
        for a in group:
            core = " [CORE]" if condition == "core" and a["id"] in CORE_ATOMS else ""
            also = [f"{x} ({ALL_ATOMS[x]['pid']})" for x in a["members"] if x != a["id"]]
            note = f"  (also stated as {', '.join(also)})" if also else ""
            lines.append(f"  {a['id']} [{a['modality']}]{core} {a['text']}{note}")
        blocks.append(f"[{pid}] {text}\n" + "\n".join(lines))
    extra = CORE_BLOCK if condition == "core" else ""
    return f"<document>\n{DOC}\n</document>\n\n<atoms>\n" + "\n".join(blocks) + f"\n</atoms>\n{extra}"


def parse(raw: str) -> dict:
    return json.loads(raw[raw.index("{") : raw.rindex("}") + 1])


def validate(m: dict) -> list[str]:
    """Structural checks on one argument map; returns error strings."""
    errors: list[str] = []
    args = {a.get("id"): a for a in m.get("arguments", [])}
    used: Counter = Counter()
    refs: Counter = Counter()
    for a in args.values():
        concl = a.get("conclusion")
        if concl not in atoms:
            errors.append(f"{a.get('id')}: conclusion {concl!r} is not an atom ID")
        else:
            used[concl] += 1
        members = [x for r in ROLES for x in a.get(r, [])]
        if not a.get("premises"):
            errors.append(f"{a.get('id')}: no premises")
        for x in members:
            if x in atoms:
                used[x] += 1
            elif x in args:
                refs[x] += 1
                if x == a.get("id"):
                    errors.append(f"{x}: references itself")
            else:
                errors.append(f"{a.get('id')}: unknown member {x!r}")
    errors += [f"atom {i} used {n} times" for i, n in used.items() if n > 1]
    errors += [f"argument {i} referenced {n} times" for i, n in refs.items() if n > 1]
    # cycle check on argument references
    state: dict[str, int] = {}

    def visit(i: str) -> None:
        if state.get(i) == 2 or i not in args:
            return
        if state.get(i) == 1:
            errors.append(f"cycle through {i}")
            return
        state[i] = 1
        for r in ROLES:
            for x in args[i].get(r, []):
                if x in args:
                    visit(x)
        state[i] = 2

    for i in args:
        visit(i)
    return errors


def run(job: tuple[str, str]) -> dict:
    condition, model = job
    folder = OUT / condition / model
    path = folder / "argument_map.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    folder.mkdir(parents=True, exist_ok=True)
    user = build_user(condition)
    backend = ClaudeAgentBackend(model=model)
    raws, errors, m = [], [], {}
    for attempt in range(2):
        prompt = user if attempt == 0 else user + f"\n\nYour previous answer had these errors; return a corrected full JSON object:\n- " + "\n- ".join(errors[:30]) + f"\n\nPrevious answer:\n{raws[-1]}"
        raws.append(backend.generate(SYSTEM, prompt, {}))
        (folder / f"raw{attempt + 1}.txt").write_text(raws[-1], encoding="utf-8")
        try:
            m = parse(raws[-1])
            errors = validate(m)
        except ValueError as exc:  # json.JSONDecodeError is a ValueError
            m, errors = {}, [f"unparseable JSON: {exc}"]
        if not errors:
            break
    result = {"map": m, "errors": errors, "attempts": len(raws), "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


jobs = [(c, m) for c in CONDITIONS for m in MODELS]
with ThreadPoolExecutor(len(jobs)) as pool:
    runs = dict(zip(jobs, pool.map(run, jobs)))

for (c, m), r in runs.items():
    cost = sum(u.get("cost_usd") or 0 for u in r["usage"])
    print(f"{c:7} {m:7} args={len(r['map'].get('arguments', []))} attempts={r['attempts']} errors={len(r['errors'])} ${cost:.3f}")

nocore  haiku   args=15 attempts=2 errors=0 $0.034
nocore  sonnet  args=4 attempts=1 errors=0 $0.096
nocore  opus    args=27 attempts=1 errors=0 $0.390
core    haiku   args=5 attempts=1 errors=0 $0.012
core    sonnet  args=5 attempts=1 errors=0 $0.097
core    opus    args=9 attempts=1 errors=0 $0.256


In [8]:
from itertools import combinations


def analyse(m: dict) -> dict:
    args = {a["id"]: a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a}
    referenced = {x for a in args.values() for r in ROLES for x in a.get(r, []) if x in args}
    roots = [i for i in args if i not in referenced]
    used = {a.get("conclusion") for a in args.values()} | {x for a in args.values() for r in ROLES for x in a.get(r, [])}
    used = {x for x in used if x in atoms}
    memo: dict[str, int] = {}

    def depth(i: str, seen=()) -> int:
        if i in seen:
            return 0
        if i not in memo:
            kids = [x for r in ROLES for x in args[i].get(r, []) if x in args]
            memo[i] = 1 + max((depth(k, (*seen, i)) for k in kids), default=0)
        return memo[i]

    root_ideas = {args[i].get("conclusion") for i in roots} & set(atoms)
    concl_ideas = {a.get("conclusion") for a in args.values()} & set(atoms)
    # Proposition ids include those of every restatement folded into the idea
    pid = lambda s: {p for x in s if x in atoms for p in atoms[x]["pids"]}
    return {
        "args": len(args),
        "roots": len(roots),
        "depth": max((depth(i) for i in args), default=0),
        "ideas_used": used,
        "props_used": pid(used),
        "root_ideas": root_ideas,
        "root_props": pid(root_ideas),
        "concl_props": pid(concl_ideas),
    }


rows = []
info = {}
for (c, m), r in runs.items():
    x = analyse(r["map"])
    info[(c, m)] = x
    core_used = set(CORE_PROPS) & x["props_used"]
    rows.append(
        {
            "cond": c,
            "model": m,
            "errors": len(r["errors"]),
            "args": x["args"],
            "roots": x["roots"],
            "depth": x["depth"],
            "ideas_used": len(x["ideas_used"]),
            "%ideas": round(100 * len(x["ideas_used"]) / len(atoms)),
            "core_props_in_map": f"{len(core_used)}/{len(CORE_PROPS)}",
            "core_props_as_conclusion": f"{len(set(CORE_PROPS) & x['concl_props'])}/{len(CORE_PROPS)}",
            "core_props_as_root": f"{len(set(CORE_PROPS) & x['root_props'])}/{len(CORE_PROPS)}",
            "root_ideas_that_are_core": f"{len(x['root_ideas'] & CORE_ATOMS)}/{len(x['root_ideas'])}",
            "cost": round(sum(u.get('cost_usd') or 0 for u in r["usage"]), 3),
        }
    )
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


def jac(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if a | b else 1.0


print("\nCross-model agreement (mean pairwise Jaccard)")
for c in CONDITIONS:
    pairs = list(combinations(MODELS, 2))
    for key in ("props_used", "root_ideas"):
        v = sum(jac(info[(c, a)][key], info[(c, b)][key]) for a, b in pairs) / len(pairs)
        print(f"  {c:7} {key:11}: {v:.2f}")

  cond  model  errors  args  roots  depth  ideas_used  %ideas core_props_in_map core_props_as_conclusion core_props_as_root root_ideas_that_are_core  cost
nocore  haiku       0    15      3      4          68      28               4/4                      4/4                3/4                      3/3 0.034
nocore sonnet       0     4      3      2          23       9               4/4                      4/4                3/4                      3/3 0.096
nocore   opus       0    27      6      4         123      50               4/4                      4/4                3/4                      3/6 0.390
  core  haiku       0     5      3      2          32      13               4/4                      4/4                3/4                      3/3 0.012
  core sonnet       0     5      5      1          29      12               4/4                      4/4                4/4                      4/5 0.097
  core   opus       0     9      3      3          62      25         

In [6]:
# v01 (no resolution) vs v02 (resolved): repeated ideas and idea-level agreement
def stats(m: dict) -> dict:
    args = [a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a]
    ids = {a["id"] for a in args}
    referenced = {x for a in args for r in ROLES for x in a.get(r, []) if x in ids}
    used = [a["conclusion"] for a in args] + [x for a in args for r in ROLES for x in a.get(r, []) if x in ALL_ATOMS]
    used = [x for x in used if x in ALL_ATOMS]
    ideas = {cluster_of[x] for x in used}
    roots = {cluster_of[a["conclusion"]] for a in args if a["id"] not in referenced and a["conclusion"] in ALL_ATOMS}
    return {"args": len(args), "atoms": len(set(used)), "ideas": ideas, "roots": roots}


cmp_rows, per_cell = [], {}
for version, base in (("v01", V01), ("v02", OUT)):
    for c in CONDITIONS:
        for m in MODELS:
            s = stats(json.loads((base / c / m / "argument_map.json").read_text(encoding="utf-8"))["map"])
            per_cell[(version, c, m)] = s
            cmp_rows.append(
                {
                    "version": version,
                    "cond": c,
                    "model": m,
                    "args": s["args"],
                    "atoms_used": s["atoms"],
                    "distinct_ideas": len(s["ideas"]),
                    "repeated_atoms": s["atoms"] - len(s["ideas"]),
                }
            )
cmp = pd.DataFrame(cmp_rows)
print(cmp.to_string(index=False))
print("\nTotals per version:")
print(cmp.groupby("version")[["args", "atoms_used", "distinct_ideas", "repeated_atoms"]].sum().to_string())

print("\nIdea-level agreement across models (mean pairwise Jaccard)")
for version in ("v01", "v02"):
    for c in CONDITIONS:
        pairs = list(combinations(MODELS, 2))
        for key in ("ideas", "roots"):
            v = sum(jac(per_cell[(version, c, a)][key], per_cell[(version, c, b)][key]) for a, b in pairs) / len(pairs)
            print(f"  {version} {c:7} {key:5}: {v:.2f}")

version   cond  model  args  atoms_used  distinct_ideas  repeated_atoms
    v01 nocore  haiku    28         107             106               1
    v01 nocore sonnet     7          36              36               0
    v01 nocore   opus    20          92              91               1
    v01   core  haiku    14          49              47               2
    v01   core sonnet     4          22              21               1
    v01   core   opus    14          85              83               2
    v02 nocore  haiku    15          68              68               0
    v02 nocore sonnet     4          23              23               0
    v02 nocore   opus    27         123             123               0
    v02   core  haiku     5          32              32               0
    v02   core sonnet     5          29              29               0
    v02   core   opus     9          62              62               0

Totals per version:
         args  atoms_used  distinct_ideas  

In [ ]:
import yaml

from incolor.argumentation.diagram import ArgumentMapDiagram

# Atom-level types from claim_core_test (opus), mapped onto the standard theme's claim types
THEME_TYPE = {"FACT": "OBSERVATION", "VIEW": "VIEW", "FORECAST": "FORECAST", "PREDICTION": "PREDICTION", "ASSUMPTION": "SUPPOSITION"}
# Atomise v01 modalities mapped onto the theme's four
THEME_MODALITY = {
    "ASSERTED": "ASSERTED",
    "ATTRIBUTED": "ASSERTED",
    "EXPECTED": "HEDGED",
    "HEDGED": "HEDGED",
    "SPECULATIVE": "SPECULATIVE",
    "SUPPOSED": "HYPOTHETICAL",
}
atom_type = {
    r["id"]: THEME_TYPE.get(r["claim_type"].upper(), "VIEW")
    for r in json.loads((ROOT / "outputs/claim_core_test/opus.json").read_text(encoding="utf-8"))["rows"]
}


def to_theme_map(m: dict) -> dict:
    """Convert a run's map to the samples argument-map schema (atoms become statements)."""
    args = [a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a]
    used = {a["conclusion"] for a in args} | {x for a in args for r in ROLES for x in a.get(r, []) if x in atoms}
    statements = [
        {
            "id": i,
            "brief": a["text"],
            "statement": a["text"],
            "claim_type": atom_type.get(i, "VIEW"),
            "modality": THEME_MODALITY[a["modality"]],
            "evidence": [i],
        }
        for i, a in atoms.items()
        if i in used
    ]
    arguments = [
        {"id": a["id"], "summary": a.get("summary", ""), "conclusion": a["conclusion"], **{r: a.get(r, []) for r in ROLES}}
        for a in args
    ]
    return {"statements": statements, "arguments": arguments}


for (c, mdl), r in runs.items():
    themed = to_theme_map(r["map"])
    folder = OUT / c / mdl
    (folder / "argument_map.yaml").write_text(yaml.safe_dump(themed, sort_keys=False, allow_unicode=True), encoding="utf-8")
    ArgumentMapDiagram(themed).save(folder / "diagram.mmd")
print("wrote", len(runs), "themed diagram.mmd + argument_map.yaml files under", OUT)


def outline(c: str, mdl: str) -> str:
    m = runs[(c, mdl)]["map"]
    args = {a["id"]: a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a}
    referenced = {x for a in args.values() for r in ROLES for x in a.get(r, []) if x in args}
    out: list[str] = []

    def show(i: str, level: int, role: str = "") -> None:
        a = args[i]
        c_id = a["conclusion"]
        tag = "*" if c_id in CORE_ATOMS else " "
        out.append(f"{'    ' * level}{tag}{role + ': ' if role else ''}{c_id} {atoms.get(c_id, {}).get('text', '?')}")
        for r in ROLES:
            for x in a.get(r, []):
                if x in args:
                    show(x, level + 1, r)
                else:
                    t = "*" if x in CORE_ATOMS else " "
                    out.append(f"{'    ' * (level + 1)}{t}{r}: {x} {atoms.get(x, {}).get('text', '?')}")

    for i in args:
        if i not in referenced:
            show(i, 0)
            out.append("")
    return "\n".join(out)

wrote 6 themed diagram.mmd + argument_map.yaml files under c:\Users\angel.adeoye\dev\financial_research\outputs\argument_core_test_v02


In [9]:
# * = atom of a core proposition
VIEW = [("nocore", "opus"), ("core", "opus")]
for c, mdl in VIEW:
    print(f"===== {c} / {mdl} =====")
    print(outline(c, mdl))

===== nocore / opus =====
*P005-1 The Fed hikes 25bp.
     premises: P012-1 The Fed is now widely expected to hike Wednesday by 25bps.
    *premises: P016-2 The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike.
         premises: P017-2 The Fed risks a sharp and disorderly UST long end move.
             premises: P018-1 US 30Y rapidly tests 5.75%.
             premises: P096-12 Treasury buybacks prove futile to stop the UST long end move.
                 premises: P096-13 Treasury buybacks stumbled out the gate.
                 premises: P020-2 Orthodoxy is not to use buybacks to quell long end rate spike.
             premises: P095-3 There is a sharp twist steepening of the curve.
                 premises: P096-5 Twist steepening comes from a lower near term policy path.
                 premises: P096-6 Twist steepening comes from a perceived very high bar to hike.
                 premises: P096-7 Twist steepening comes from long end rates.
       